[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/google-adk/google_adk_financial_advisor.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

本 Notebook 演示如何使用金融顾问智能体，并通过 **Phoenix 追踪获得完整可观测性**。智能体通过一组专业 AI 子智能体提供综合金融建议：

1. **数据分析智能体**：为指定股票代码生成市场分析报告
2. **交易分析智能体**：根据风险承受能力和投资期限制定交易策略
3. **执行智能体**：制定交易策略实施计划
4. **风险评估智能体**：分析风险并提供缓解策略

**🔍 Phoenix 追踪**：所有智能体交互、工具使用和 LLM 调用都会自动追踪并发送到 Phoenix，用于可观测性、性能监控与评估。

**⚠️ 重要声明**：本示例仅供教学，不构成金融建议。

✅ 运行本 Notebook 需要启用 Vertex AI 的 Google Cloud 账号，也可以使用 Phoenix Cloud 账号。


# 设置密钥与依赖

## 安装依赖

In [ ]:
%pip install "google-cloud-aiplatform[adk,agent-engines]>=1.93.0" "google-genai>=1.9.0" "pydantic>=2.10.6" "python-dotenv>=1.0.1" "google-adk>=1.0.0" "openinference-instrumentation-google-adk" "arize-phoenix-otel"

## 设置 Google Cloud

金融顾问智能体需要配置 Google Cloud，因为它使用：
- **Vertex AI**：调用 Gemini 2.5 Pro 模型
- **Google Search** 工具：分析市场数据

1. **创建 Google Cloud 项目**（如果尚未创建）
2. **启用所需 API：**
   - Vertex AI API
   - Generative AI API

3. **设置身份验证：**
   ```bash
   gcloud auth application-default login
   gcloud auth application-default set-quota-project YOUR-PROJECT-ID
   ```

## 环境变量

在本目录创建 `.env` 文件，包含以下变量：

```bash
# Google ADK 必填配置
GOOGLE_GENAI_USE_VERTEXAI=1
GOOGLE_CLOUD_PROJECT=your-project-id
GOOGLE_CLOUD_LOCATION=your-project-location  # 例如 us-central1

# 可选：Phoenix Cloud 配置（用于生产环境）
# PHOENIX_API_KEY=your-phoenix-api-key
# PHOENIX_COLLECTOR_ENDPOINT=your-phoenix-hostname
```。

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

# Google Cloud 身份验证：ADK 智能体必需
if not (google_cloud_project := os.getenv("GOOGLE_CLOUD_PROJECT")):
    google_cloud_project = getpass("🔑 请输入 Google Cloud 项目 ID： ")
os.environ["GOOGLE_CLOUD_PROJECT"] = google_cloud_project

if not (google_cloud_location := os.getenv("GOOGLE_CLOUD_LOCATION")):
    google_cloud_location = getpass("🔑 请输入 Google Cloud 区域（例如 us-central1）： ")
os.environ["GOOGLE_CLOUD_LOCATION"] = google_cloud_location

# 设置 Vertex AI 必需的环境变量
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"

# 可选：Phoenix Cloud 配置
# 未设置时，追踪发送到本地 Phoenix 实例
if phoenix_api_key := os.getenv("PHOENIX_API_KEY"):
    print("✅ 已找到 Phoenix API 密钥，追踪将发送到 Phoenix Cloud")
    if phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT"):
        print(f"✅ 使用 Phoenix 端点： {phoenix_endpoint}")
else:
    print("📍 未找到 Phoenix API 密钥，追踪将发送到本地 Phoenix 实例")
    print(
        "💡 若要使用 Phoenix Cloud，请设置 PHOENIX_API_KEY 和 PHOENIX_COLLECTOR_ENDPOINT 环境变量"
    )

# 配置追踪

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from phoenix.otel import register

# 配置 Phoenix 追踪器
tracer_provider = register(
    project_name="google-adk-financial-advisor",  # 用于组织追踪的项目名称
    auto_instrument=True,  # 根据已安装的 OI 依赖自动为应用插桩
)

# 构建金融顾问系统

这里直接定义所有提示词与智能体，使本 Notebook 可以独立运行。

In [ ]:
# 定义主协调器提示词
FINANCIAL_COORDINATOR_PROMPT = """
角色：担任专业金融咨询助手。
你的主要目标是协调一组专家子智能体，引导用户按结构化流程获取金融建议。
帮助用户分析市场股票代码、制定交易策略、确定执行计划，并评估整体风险。

交互的总体指令：

首先向用户介绍自己，例如：

“你好！我将帮助你了解金融决策。
我的主要目标是通过分步流程为你提供综合金融建议。
我们会一起分析市场股票代码、制定有效的交易策略、确定清晰的执行计划，
并全面评估整体风险。

请记住，每一步都可以要求‘以 Markdown 显示详细结果’。

准备好开始了吗？”

然后立即显示以下免责声明：

“重要声明：仅供教学与信息参考。
本工具提供的信息与交易策略概要，包括分析、评论或潜在情景，均由 AI 模型生成，
仅供教学和信息参考。
它们不构成，也不应被理解为金融建议、投资推荐、背书，
或买卖任何证券或其他金融工具的要约。
Google 及其关联公司不对所提供信息的完整性、准确性、可靠性、适用性或可用性
作出任何明示或暗示的陈述或保证。依赖这些信息的风险完全由你自行承担。
这不是买卖任何证券的要约。
不应仅根据这里提供的信息作出投资决定。
金融市场存在风险，过往表现不代表未来结果。
在作出任何投资决定前，应自行进行充分研究，并咨询合格的独立金融顾问。
使用本工具并查看这些策略，即表示你理解本声明，并同意 Google 及其关联公司
不对因使用或依赖这些信息而造成的任何损失或损害负责。”

每一步都应清楚告知用户当前调用的子智能体，以及需要用户提供的具体信息。
各子智能体完成任务后，解释其输出及其对整体金融咨询流程的作用。
确保正确使用所有状态键，在子智能体之间传递信息。
以下是分步流程。
每一步都必须显式调用指定子智能体，并严格遵守指定的输入输出格式：

* 收集市场数据分析（子智能体：data_analyst_agent）
* 制定交易策略（子智能体：trading_analyst_agent）
* 确定最佳执行策略（子智能体：execution_analyst_agent）
* 评估整体风险状况（子智能体：risk_analyst_agent）
"""

In [ ]:
# 数据分析智能体提示词
DATA_ANALYST_PROMPT = """
智能体角色：data_analyst
工具使用：仅使用 Google Search 工具。

总体目标：为 provided_ticker 生成全面且及时的市场分析报告。通过反复调用 Google Search 工具，收集目标数量的独立、近期（在指定时间范围内）且有洞察价值的信息。分析同时关注 SEC 相关数据与一般市场或股票情报，再仅基于已收集数据综合为结构化报告。

输入（来自调用智能体或环境）：
- provided_ticker：（字符串，必填）股票代码，例如 AAPL、GOOGL、MSFT。data_analyst 智能体不得向用户索取此输入。
- max_data_age_days：（整数，可选，默认 7）判断信息是否新鲜且相关的最大天数。
- target_results_count：（整数，可选，默认 10）用于支撑分析的独立、高质量搜索结果的目标数量。

预期最终输出（结构化报告）：
data_analyst 必须返回一份完整综合报告，结构如下：

**[provided_ticker] 市场分析报告**
**报告日期：** [报告生成当日日期]
**信息时效目标：** 数据主要来自最近 [max_data_age_days] 天。
**查阅的独立一手来源数量：** [实际数量]

**1. 执行摘要：**
   * 仅根据已收集数据，以 3–5 个要点简述最关键发现与整体展望。

**2. 近期 SEC 申报与监管信息：**
   * 总结近期 SEC 申报中的关键信息。
   * 如果未发现重要的近期 SEC 申报，应明确说明。

**3. 近期新闻、股价表现背景与市场情绪：**
   * **重大新闻：** 总结影响公司或股票的重大新闻。
   * **股价表现背景：** 简述近期股价趋势。
   * **市场情绪：** 说明主导情绪并简述理由。

**4. 近期分析师评论与展望：**
   * 总结近期分析师评级、目标价调整及主要理由。
   * 如果未发现重要的近期分析师评论，应明确说明。

**5. 关键风险与机会（根据已收集数据得出）：**
   * **已识别风险：** 以要点列出关键风险因素。
   * **已识别机会：** 以要点列出潜在机会。

**6. 主要参考文章：**
   * 列出来源标题、URL、发布日期与简要相关性说明。
"""

In [ ]:
# 交易分析智能体提示词
TRADING_ANALYST_PROMPT = """
制定定制交易策略（子智能体：trading_analyst）

trading_analyst 的总体目标：
批判性评估全面的 market_data_analysis_output，构思并概述至少五种不同的交易策略。
每种策略都必须针对用户声明的风险态度与预期投资期限进行定制。

输入（传给 trading_analyst）：
- 用户风险态度（user_risk_attitude）：Conservative、Moderate 或 Aggressive
- 用户投资期限（user_investment_period）：Short-term、Medium-term 或 Long-term
- 市场分析数据（来自状态）：market_data_analysis_output

预期输出：
生成至少 5 种不同的交易策略，每种包含：
1. 策略名称与描述
2. 基于市场分析的依据
3. 与用户风险状况和期限的匹配程度
4. 需要监控的关键指标
5. 入场与退出条件
6. 该策略的具体风险

输出为详细 Markdown 报告，每种策略都有清晰章节。

重要声明：加入标准金融免责声明，说明内容仅供教学，不构成金融建议。
"""

In [ ]:
# 执行分析智能体提示词
EXECUTION_ANALYST_PROMPT = """
确定最佳执行策略（子智能体：execution_analyst）

总体目标：
为实施拟议交易策略创建全面且可操作的执行计划。计划必须准确匹配用户的风险承受能力、投资期限和执行偏好。

输入：
- proposed_trading_strategies_output（来自状态）：交易分析师制定的交易策略
- user_risk_attitude：用户风险承受水平
- user_investment_period：用户投资期限
- user_execution_preferences（可选）：特定经纪商、订单类型或执行偏好

预期输出：
生成详细执行计划，涵盖：

**I. 基础执行理念**
- 目标与用户状况的匹配
- 风险管理方式
- 成本控制重点

**II. 入场执行策略**
- 最佳入场条件与时机
- 推荐订单类型与下单方式
- 初始仓位规模与风险分配
- 初始止损策略

**III. 持仓与交易中管理策略**
- 主动监控或被动持仓方式
- 动态风险管理（止损调整）
- 应对波动与回撤

**IV. 累积加仓策略**
- 加仓条件与理由
- 分批加仓的执行方法
- 仓位增大后的风险调整

**V. 部分卖出（止盈）策略**
- 部分退出的触发条件与理由
- 分批减仓的执行方法
- 剩余仓位管理

**VI. 全面退出策略**
- 完全平仓的条件
- 订单类型与执行方式
- 滑点和市场影响考量

输出为详细 Markdown 报告，各策略类型都有清晰章节。

重要：加入标准金融免责声明。
"""

In [ ]:
# 风险分析智能体提示词
RISK_ANALYST_PROMPT = """
评估整体风险状况（子智能体：risk_analyst）

总体目标：
全面分析拟议金融计划的风险，评估市场分析、交易策略、执行计划与用户声明的风险承受能力和投资期限之间的一致性。

输入：
- market_data_analysis_output（来自状态）：市场分析数据
- proposed_trading_strategies_output（来自状态）：拟议交易策略
- execution_plan_output（来自状态）：执行计划
- user_risk_attitude：用户声明的风险承受能力
- user_investment_period：用户声明的投资期限

预期输出：
生成全面风险评估报告，涵盖：

**1. 风险执行摘要**
- 已识别的主要风险因素
- 整体定性风险评估
- 与用户风险状况的匹配程度

**2. 市场风险**
- 已识别的市场相关风险
- 潜在影响评估
- 缓解策略

**3. 流动性风险**
- 流动性问题评估
- 缓解方法

**4. 交易对手与平台风险**
- 经纪商与平台相关风险
- 推荐保障措施

**5. 运营与技术风险**
- 执行和运营风险
- 人为失误因素
- 技术相关问题

**6. 策略特有风险与模型风险**
- 所选策略的特有风险
- 模型与假设风险
- 集中度风险

**7. 交易者心理风险**
- 行为与情绪风险
- 纪律与执行一致性挑战

**8. 与用户状况的整体匹配及结语**
- 与用户声明偏好的匹配评估
- 潜在不匹配或问题
- 关键考量与建议

输出为详细 Markdown 报告，清楚区分风险类别并提供可操作建议。

重要：加入完整金融免责声明。
"""

In [ ]:
# 创建子智能体
MODEL = "gemini-2.5-pro"

# 数据分析智能体
data_analyst_agent = LlmAgent(
    model=MODEL,
    name="data_analyst_agent",
    description="使用 Google Search 分析市场数据并创建综合市场报告",
    instruction=DATA_ANALYST_PROMPT,
    output_key="market_data_analysis_output",
    tools=[google_search],
)

# 交易分析智能体
trading_analyst_agent = LlmAgent(
    model=MODEL,
    name="trading_analyst_agent",
    description="根据市场分析与用户偏好制定定制交易策略",
    instruction=TRADING_ANALYST_PROMPT,
    output_key="proposed_trading_strategies_output",
)

# 执行分析智能体
execution_analyst_agent = LlmAgent(
    model=MODEL,
    name="execution_analyst_agent",
    description="为实施交易策略创建详细执行计划",
    instruction=EXECUTION_ANALYST_PROMPT,
    output_key="execution_plan_output",
)

# 风险分析智能体
risk_analyst_agent = LlmAgent(
    model=MODEL,
    name="risk_analyst_agent",
    description="全面分析和评估金融计划的风险",
    instruction=RISK_ANALYST_PROMPT,
    output_key="final_risk_assessment_output",
)

print("✓ 所有子智能体创建成功！")

In [ ]:
# 创建金融主协调智能体
financial_coordinator = LlmAgent(
    name="financial_coordinator",
    model=MODEL,
    description=(
        "引导用户按结构化流程获取金融"
        "建议，协调一组专家子智能体，帮助"
        "分析市场股票代码、制定交易策略、确定"
        "执行计划并评估整体风险。"
    ),
    instruction=FINANCIAL_COORDINATOR_PROMPT,
    output_key="financial_coordinator_output",
    tools=[
        AgentTool(agent=data_analyst_agent),
        AgentTool(agent=trading_analyst_agent),
        AgentTool(agent=execution_analyst_agent),
        AgentTool(agent=risk_analyst_agent),
    ],
)

print("✓ 金融协调智能体创建成功！")
print(f"智能体名称： {financial_coordinator.name}")
print(f"模型： {financial_coordinator.model}")
print(f"子智能体数量： {len(financial_coordinator.tools)}")

# 测试金融顾问智能体

以下辅助函数支持在 Jupyter Notebook 中与金融顾问智能体交互，负责智能体初始化、会话管理，并提供异步封装以查询智能体和显示回复。


In [ ]:
from google.adk.runners import InMemoryRunner
from google.genai.types import Part, UserContent

# 仅初始化一次运行器与会话
runner = None
session = None


async def initialize_runner():
    """初始化智能体的运行器与会话。"""
    global runner, session
    if runner is None:
        runner = InMemoryRunner(agent=financial_coordinator)
        session = await runner.session_service.create_session(
            app_name=runner.app_name, user_id="demo_user"
        )
    return runner, session


async def run_agent_query_async(query: str):
    """异步辅助函数，用于向金融顾问智能体发送查询。"""
    try:
        runner, session = await initialize_runner()
        content = UserContent(parts=[Part(text=query)])

        response_parts = []

        # 通过适当的错误处理改善异步迭代
        async_iterator = runner.run_async(
            user_id=session.user_id,
            session_id=session.id,
            new_message=content,
        )

        try:
            async for event in async_iterator:
                # 更全面地检查事件内容
                if event and hasattr(event, "content") and event.content:
                    if hasattr(event.content, "parts") and event.content.parts:
                        if (
                            len(event.content.parts) > 0
                            and event.content.parts[0]
                            and hasattr(event.content.parts[0], "text")
                            and event.content.parts[0].text
                        ):
                            response_parts.append(event.content.parts[0].text)
        except GeneratorExit:
            # 妥善处理生成器退出（常见于 OpenTelemetry 上下文问题）
            pass
        except Exception as iter_error:
            print(f"警告：迭代时发生错误： {iter_error}")
            # 继续使用当前已收集的内容

        result = "".join(response_parts) if response_parts else "未生成回复"
        return result

    except Exception as e:
        import traceback

        print(f"run_agent_query_async 发生错误： {str(e)}")
        traceback.print_exc()
        return f"Error: {str(e)}"


# 供 Jupyter 使用的简单封装：通过 await 调用
async def query_agent(query: str):
    """查询智能体的简单函数，使用 'await query_agent(your_question)' 调用。"""
    response = await run_agent_query_async(query)
    return response


def print_response(title: str, response: str, max_length: int = None):
    """打印格式化回复的辅助函数。"""
    print(f"\n{'=' * 60}")
    print(f"{title}")
    print(f"{'=' * 60}")

    if max_length and len(response) > max_length:
        print(response[:max_length] + "\n\n... [为便于显示，回复已截断]")
    else:
        print(response)
    print(f"{'=' * 60}\n")

## 📊 Phoenix 追踪面板

开始运行下方示例后，即可在 Phoenix 中查看追踪：

**本地 Phoenix 实例：**
- 访问：http://localhost:6006
- 在智能体执行时实时查看追踪
- 分析性能指标与令牌用量

**Phoenix Cloud：**
- 访问 Phoenix Cloud 面板
- 使用高级分析与协作功能

**Phoenix 中可查看的内容：**
- 🔗 **追踪树**：从用户查询到最终回复的完整执行流程
- ⏱️ **耗时数据**：每次智能体与工具调用的延迟
- 💰 **令牌用量**：输入、输出令牌与成本跟踪
- 🔍 **工具调用**：Google Search 与智能体交互的详细日志
- 📊 **性能指标**：所有运行的汇总统计

这些追踪有助于调试、优化，并了解多智能体系统的实际表现。


In [ ]:
# 示例 1：智能体介绍
print("🤖 开始介绍智能体……")
introduction_query = "Hello, who are you and what can you help me with?"
introduction_response = await query_agent(introduction_query)
print_response("✅ 智能体介绍", introduction_response, max_length=1500)

In [ ]:
# 示例 2：基础股票分析
print("📊 开始分析苹果股票……")
stock_analysis_query = (
    "I'd like to analyze Apple stock (AAPL). Can you provide a comprehensive market analysis?"
)
stock_analysis_response = await query_agent(stock_analysis_query)
print_response("✅ 苹果股票分析", stock_analysis_response, max_length=2000)

In [ ]:
# 示例 3：完整金融咨询流程
print("💼 开始综合分析微软……")
comprehensive_query = """
I'm interested in investing in Microsoft (MSFT). Here are my preferences:
- Risk tolerance: Moderate
- Investment period: Long-term (5+ years)
- Goal: Build wealth for retirement

Please provide a complete analysis including market data, trading strategies,
execution plan, and risk assessment.
"""

comprehensive_response = await query_agent(comprehensive_query.strip())
print_response("✅ 微软综合分析", comprehensive_response, max_length=3000)

In [ ]:
# 示例 4：交互式会话
print("🔄 开始交互式特斯拉分析……")

# 第 1 步：提供股票代码
step1_query = "I want to analyze Tesla (TSLA)"
step1_response = await query_agent(step1_query)
print_response("✅ 第 1 步：特斯拉股票代码请求", step1_response, max_length=1000)

# 第 2 步：提供偏好
step2_query = "My risk attitude is aggressive and my investment period is medium-term (2-3 years)"
step2_response = await query_agent(step2_query)
print_response("✅ 第 2 步：风险与期限偏好", step2_response, max_length=1500)

In [ ]:
# 🎯 自定义查询界面：修改后运行！
# ✏️ 修改下方查询以提出你自己的问题

custom_query = """
I'm a beginner investor with $10,000 to invest. I'm 25 years old and want to start building
wealth for the long term. I have a moderate risk tolerance. What would you recommend for
getting started with stock investing?
"""

print("🎯 正在运行自定义查询……")
custom_response = await query_agent(custom_query.strip())
print_response("✅ 自定义查询回复", custom_response, max_length=2000)